In [1]:
#1. Setup

import os, re, time, random
from dotenv import load_dotenv
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.metrics.pairwise import cosine_similarity
from cerebras.cloud.sdk import Cerebras

# Load environment variables
load_dotenv()

# Configs
ROLE_MAP = {"A": "Abuser", "V": "Victim", "T": "Third"}
ROLE_LABELS = ["Abuser", "Victim", "Third"]
SEED = 42
TEST_SIZE = 200
MODEL_NAME  = "llama3.1-8b" 
REQUEST_DELAY_SECONDS = 0.3


# Cerebras client
client = Cerebras(
    api_key=os.environ.get("CEREBRAS_API_KEY")
)

# preprocessing
def resolve_data_path() -> Path:
    candidates = [
        Path.cwd() / "reviewer.csv",
        Path.cwd() / "HWK2" / "reviewer.csv",
        Path.cwd().parent / "HWK2" / "reviewer.csv",
    ]
    for p in candidates:
        if p.exists():
            return p
    raise FileNotFoundError("can't find reviewer.csv, please check the path.")

DATA_PATH = resolve_data_path()
print("Data path :", DATA_PATH)
print("Model     :", MODEL_NAME)
print("API key   : OK" if os.getenv("CEREBRAS_API_KEY") else "API key   : MISSING")
print("API key prefix:", os.environ.get("CEREBRAS_API_KEY")[:6])

c:\Users\alvin\OneDrive\Documents\VS Code Projects\Natural Language Processing\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Data path : c:\Users\alvin\OneDrive\Documents\VS Code Projects\Natural Language Processing\HWK2\reviewer.csv
Model     : llama3.1-8b
API key   : OK
API key prefix: csk-5t


In [2]:
#2. load reviewer.csv, map roles, stratified split

def clean_text(text: str) -> str:
    return re.sub(r"\s+", " ", str(text)).strip()

df_raw = pd.read_csv(DATA_PATH)

role_col = "Victim/Abuser/Third"
text_col = "body"

df = df_raw[[text_col, role_col]].dropna().copy()
df[role_col] = df[role_col].map(ROLE_MAP).fillna(df[role_col])
df["text"] = df[text_col].apply(clean_text)
df = df[df[role_col].isin(ROLE_LABELS)].reset_index(drop=True)

print(f"Total labeled rows: {len(df)}")

# Stratified split
source_df, test_df = train_test_split(
    df,
    test_size=200,
    stratify=df[role_col],
    random_state=SEED,
)


source_df = source_df.reset_index(drop=True)
test_df   = test_df.reset_index(drop=True)

print(f"Source set : {len(source_df)} rows")
print(f"Test set   : {len(test_df)} rows\n")

print("Source role distribution:")
print(source_df[role_col].value_counts(), "\n")
print("Test role distribution:")
print(test_df[role_col].value_counts())

Total labeled rows: 403
Source set : 203 rows
Test set   : 200 rows

Source role distribution:
Victim/Abuser/Third
Abuser    112
Victim     61
Third      30
Name: count, dtype: int64 

Test role distribution:
Victim/Abuser/Third
Abuser    111
Victim     60
Third      29
Name: count, dtype: int64


In [3]:
#3. helper: call Cerebras chat API

def call_cerebras_chat(prompt: str, max_tokens: int = 64) -> str:

    resp = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {"role": "user", "content": prompt},
        ],
        max_tokens=max_tokens,
        temperature=0.0,
    )
    return resp.choices[0].message.content.strip()

In [4]:
#6. handpicked examples for few-shot (from reviewer.csv)

handpicked_examples = [
    {
        "role": "Abuser",
        "text": "i can spy on my child whenever i want its amazing he cant go anywhere without me knowing look"
    },
    {
        "role": "Abuser",
        "text": "Now that I can spy on my wife I'll always know when she's cheating. Thank you for the divorce"
    },
    {
        "role": "Victim",
        "text": "I hate this app it's an app used to stalk kids. Let them live there lives, whoever made this app should be ashamed of themselves."
    },
    {
        "role": "Victim",
        "text": "This app literally ruined my teenage years I have absolutely no privacy my parents are always stalking me now for no reason."
    },
    {
        "role": "Third",
        "text": "Parents use this app to stalk their kids, this app should be illegal."
    },
    {
        "role": "Third",
        "text": "This app is useful in regards of keeping track of people if they are lost or get kidnapped. HOWEVER, I do not condone this app to be used to stalk your child and constantly keep tabs on them. It is all about trust."
    },
]

len(handpicked_examples)

6

In [5]:
#7. build few-shot prompt

def build_fewshot_prompt(text: str, examples, K: int) -> str:
    exs = examples[:K]

    parts = [
        "You are a classifier that reads app reviews about surveillance or tracking apps.",
        "Your task is to decide the narrator's role.",
        "",
        "Roles:",
        "- Abuser: someone using the app to control, monitor, or stalk others.",
        "- Victim: someone being monitored, stalked, or harmed by others.",
        "- Third-party: a neutral reviewer, observer, or bystander.",
        "",
        "Here are labeled examples:\n",
    ]

    for i, ex in enumerate(exs, 1):
        parts.append(f"Example {i}:")
        parts.append(f"Review: {ex['text']}")
        parts.append(f"Role: {ex['role']}\n")

    parts.append("Now classify the new review below.")
    parts.append("Review:")
    parts.append(text)
    parts.append("Answer with exactly one word: Abuser, Victim, or Third.")

    return "\n".join(parts)

In [6]:
#8. few-shot classifier using handpicked examples

def classify_role_few_shot(text: str, examples, K: int) -> str:
    
    prompt = build_fewshot_prompt(text, examples, K)
    raw = call_cerebras_chat(prompt)
    ans = raw.strip()

    for r in ROLE_LABELS:
        if r.lower() in ans.lower():
            return r
    return "Unknown"

In [7]:
#9. run handpicked few-shot for K=1..5 on all test samples

for K in range(1, 6):
    preds = []
    for txt in tqdm(test_df["text"], desc=f"Handpicked few-shot K={K}"):
        pred = classify_role_few_shot(txt, handpicked_examples, K=K)
        preds.append(pred)
        time.sleep(0.1)

    col_name = f"pred_hand_k{K}"
    test_df[col_name] = preds
    print(f"Finished {col_name}")

Handpicked few-shot K=1: 100%|██████████| 200/200 [06:06<00:00,  1.83s/it]


Finished pred_hand_k1


Handpicked few-shot K=2: 100%|██████████| 200/200 [11:01<00:00,  3.31s/it]


Finished pred_hand_k2


Handpicked few-shot K=3: 100%|██████████| 200/200 [09:52<00:00,  2.96s/it]


Finished pred_hand_k3


Handpicked few-shot K=4: 100%|██████████| 200/200 [08:53<00:00,  2.67s/it]


Finished pred_hand_k4


Handpicked few-shot K=5: 100%|██████████| 200/200 [06:48<00:00,  2.04s/it]

Finished pred_hand_k5


In [10]:
from sklearn.metrics import classification_report

print("="*80)
print("HANDPICKED FEW-SHOT CLASSIFICATION RESULTS")
print("="*80)

for K in range(1, 6):
    col_name = f"pred_hand_k{K}"
    print(f"\n### Handpicked Few-Shot K={K} ###\n")
    print(classification_report(
        test_df["Victim/Abuser/Third"],
        test_df[col_name],
        labels=ROLE_LABELS,
        zero_division=0,
    ))

# Save handpicked predictions to CSV
# Check if all required columns exist
required_cols = [f"pred_hand_k{K}" for K in range(1, 6)]
missing_cols = [c for c in required_cols if c not in test_df.columns]

if missing_cols:
    print(f"Warning: Missing columns {missing_cols}. Run cell #5 first to generate predictions.")
else:
    out_df = test_df[["text", "Victim/Abuser/Third"]].copy()
    out_df.columns = ["text", "role"]
    out_df["idx"] = range(len(out_df))
    for K in range(1, 6):
        out_df[f"pred_hand_k{K}"] = test_df[f"pred_hand_k{K}"]
    out_df.to_csv("handpicked_predictions.csv", index=False)
    print(f"\nSaved {len(out_df)} predictions to handpicked_predictions.csv")

HANDPICKED FEW-SHOT CLASSIFICATION RESULTS

### Handpicked Few-Shot K=1 ###

              precision    recall  f1-score   support

      Abuser       0.72      0.90      0.80       111
      Victim       0.84      0.52      0.64        60
       Third       0.40      0.34      0.37        29

    accuracy                           0.70       200
   macro avg       0.65      0.59      0.60       200
weighted avg       0.71      0.70      0.69       200


### Handpicked Few-Shot K=2 ###

              precision    recall  f1-score   support

      Abuser       0.69      0.86      0.77       111
      Victim       0.85      0.47      0.60        60
       Third       0.29      0.28      0.28        29

    accuracy                           0.66       200
   macro avg       0.61      0.54      0.55       200
weighted avg       0.68      0.66      0.65       200


### Handpicked Few-Shot K=3 ###

              precision    recall  f1-score   support

      Abuser       0.72      0.93     